## **RedPanda RVC**
A voice conversion tool based on Applio, with RedPanda-specific training and WebUI improvements.

[GitHub](https://github.com/redpanda343/redpanda-rvc)

<br>

### **Credits**
- Encryption method: [Hina](https://github.com/hinabl)
- Uv code: [Shirou](https://github.com/ShiromiyaG)
- Filebrowser Login Password Fix & Tunnels code: [Nick088](https://linktr.ee/Nick088)
- Based on [Applio](https://github.com/IAHispano/Applio)

## Install

In [ ]:
import codecs
import os
import shutil
import subprocess
import sys
from pathlib import Path
from IPython.display import clear_output
rot_47 = lambda encoded_text: "".join(
    [
        (
            chr(
                (ord(c) - (ord("a") if c.islower() else ord("A")) - 47) % 26
                + (ord("a") if c.islower() else ord("A"))
            )
            if c.isalpha()
            else c
        )
        for c in encoded_text
    ]
)

new_name = rot_47("kmjbmvh_hg")
uioawhd = rot_47(codecs.decode("pbbxa://oqbpcj.kwu/zmlxivli343/zmlxivli-zdk.oqb", "rot_13"))
repo_dir = Path("/kaggle/working") / new_name

subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
if repo_dir.exists():
    if not (repo_dir / ".git").is_dir():
        raise RuntimeError(f"{repo_dir} exists but is not a Git repository.")
    origin = subprocess.run(
        ["git", "-C", str(repo_dir), "remote", "get-url", "origin"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if origin.rstrip("/").removesuffix(".git") != uioawhd.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Existing repository has an unexpected origin: {origin}")
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", uioawhd, str(repo_dir)], check=True)

subprocess.run(["apt", "update", "-y"], check=True)
subprocess.run(["apt", "install", "-y", "portaudio19-dev", "psmisc"], check=True)
uv_executable = shutil.which("uv") or str(Path(sys.executable).with_name("uv"))
subprocess.run(
    [
        uv_executable,
        "pip",
        "install",
        "-q",
        "-r",
        str(repo_dir / "requirements.txt"),
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu128",
        "--index-strategy",
        "unsafe-best-match",
        "--system",
    ],
    check=True,
)
os.chdir(repo_dir)
subprocess.run(
    [sys.executable, "core.py", "prerequisites", "--models", "--exe", "--pretraineds-hifigan"],
    check=True,
)
if shutil.which("filebrowser") is None:
    subprocess.run(
        [
            "bash",
            "-o",
            "pipefail",
            "-c",
            "curl -fsSL https://raw.githubusercontent.com/filebrowser/get/master/get.sh | sudo bash",
        ],
        check=True,
    )
clear_output()
print("Finished")

## Start

### Access Setup

TensorBoard is built into the RedPanda WebUI. Open the **TensorBoard** tab and click **Launch TensorBoard**. It uses the same authenticated WebUI connection, so it does not need its own tunnel or public URL.

Keep the notebook private while these services are running, and do not publish saved cell output containing the generated credentials.

Select how to expose the WebUI and FileBrowser:

- Gradio + LocalTunnel: Gradio provides the RedPanda WebUI link, including TensorBoard. LocalTunnel provides the FileBrowser link. The notebook prints separate WebUI and FileBrowser credentials plus the LocalTunnel consent password.

- LocalTunnel: LocalTunnel provides separate RedPanda WebUI and FileBrowser links. TensorBoard is inside the WebUI link. Enter the LocalTunnel consent password, then use the printed credentials for the WebUI or FileBrowser.

- Horizon: Enter the Horizon ID from https://hrzn.run/dashboard/. Approve the CLI token request when prompted. Horizon provides separate RedPanda WebUI and FileBrowser links, and the notebook prints credentials for both services. TensorBoard is inside the WebUI link.

In [ ]:
import os
import re
import secrets
import shutil
import socket
import subprocess
import sys
import threading
import time
import urllib.request
from pathlib import Path
from IPython.display import clear_output


Tunnel = "Gradio + LocalTunnel" #@param ["Gradio + LocalTunnel", "LocalTunnel", "Horizon"]
horizon_id = "" #@param {type:"string"}
repo_dir = Path("/kaggle/working/program_ml")
filebrowser_database = repo_dir / "filebrowser.db"
filebrowser_username = "applio"
filebrowser_password = secrets.token_urlsafe(24)
app_username = "applio"
app_password = secrets.token_urlsafe(24)
localtunnel_pattern = r"https://[a-zA-Z0-9.-]+\.(?:loca\.lt|localtunnel\.me)"
horizon_pattern = r"https://[a-zA-Z0-9.-]+\.hrzn\.run"
tunnel_processes = []
tunnel_logs = []
filebrowser_process = None
filebrowser_log = None
app_process = None
app_output_thread = None


def log_detail(path):
    try:
        return path.read_text(encoding="utf-8", errors="replace")[-1000:].strip() or "no output"
    except OSError:
        return "no output"


def wait_for_port(host, port, process, log_path=None, timeout=180):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if process.poll() is not None:
            detail = log_detail(log_path) if log_path else f"exit code {process.returncode}"
            raise RuntimeError(f"Service stopped before opening port {port}: {detail}")
        try:
            with socket.create_connection((host, port), timeout=1):
                return
        except OSError:
            time.sleep(1)
    raise RuntimeError(f"Service did not open port {port} within {timeout} seconds.")


def wait_for_tunnel_url(output_file, pattern, process, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        output = output_file.read_text(encoding="utf-8", errors="replace") if output_file.exists() else ""
        match = re.search(pattern, output)
        if match:
            return match.group(0)
        if process.poll() is not None:
            raise RuntimeError(f"Tunnel stopped before providing a URL: {output[-1000:].strip() or 'no output'}")
        time.sleep(1)
    raise RuntimeError(f"Tunnel did not provide a URL: {log_detail(output_file)}")


def start_tunnel(command, name, pattern):
    output_file = repo_dir / f"{name}.log"
    log_file = open(output_file, "w", encoding="utf-8")
    process = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT, text=True)
    tunnel_processes.append(process)
    tunnel_logs.append(log_file)
    return wait_for_tunnel_url(output_file, pattern, process)


def stop_process(process):
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)


def stream_process_output(process):
    for line in process.stdout:
        print(line, end="", flush=True)


def localtunnel_password():
    last_error = None
    for _ in range(3):
        try:
            with urllib.request.urlopen("https://loca.lt/mytunnelpassword", timeout=15) as response:
                password = response.read().decode("utf-8").strip()
            if password:
                return password
        except OSError as error:
            last_error = error
        time.sleep(2)
    raise RuntimeError(f"Could not retrieve the LocalTunnel password: {last_error}")


os.chdir(repo_dir)
for pattern in (
    "lt .*--port 6969",
    "lt .*--port 9876",
    "hrzn tunnel http://localhost:6969",
    "hrzn tunnel http://localhost:9876",
):
    if shutil.which("pkill"):
        subprocess.run(["pkill", "-f", pattern], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for port in ("6969/tcp", "9876/tcp"):
    subprocess.run(["fuser", "-k", port], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if filebrowser_database.exists():
    subprocess.run(
        ["filebrowser", "config", "set", "--auth.method=json", "--database", str(filebrowser_database)],
        check=True,
    )
else:
    subprocess.run(
        ["filebrowser", "config", "init", "--auth.method=json", "--database", str(filebrowser_database)],
        check=True,
    )
update_user = subprocess.run(
    [
        "filebrowser",
        "users",
        "update",
        filebrowser_username,
        "--password",
        filebrowser_password,
        "--perm.admin",
        "--database",
        str(filebrowser_database),
    ],
    capture_output=True,
    text=True,
)
if update_user.returncode != 0:
    add_user = subprocess.run(
        [
            "filebrowser",
            "users",
            "add",
            filebrowser_username,
            filebrowser_password,
            "--perm.admin",
            "--database",
            str(filebrowser_database),
        ],
        capture_output=True,
        text=True,
    )
    if add_user.returncode != 0:
        detail = (update_user.stderr + add_user.stderr).strip() or "unknown FileBrowser error"
        raise RuntimeError(f"Could not configure the FileBrowser user: {detail}")

if Tunnel in ("Gradio + LocalTunnel", "LocalTunnel"):
    subprocess.run(["npm", "install", "-g", "localtunnel"], check=True)
elif Tunnel == "Horizon":
    if not horizon_id.strip():
        raise ValueError("horizon_id is required when Horizon is selected.")
    subprocess.run(["npm", "install", "-g", "@hrzn/cli"], check=True)
    subprocess.run(["hrzn", "login", horizon_id], check=True)
else:
    raise ValueError(f"Unsupported tunnel option: {Tunnel}")

clear_output()
print(f"RedPanda WebUI Username: {app_username}")
print(f"RedPanda WebUI Password: {app_password}")
print(f"FileBrowser Username: {filebrowser_username}")
print(f"FileBrowser Password: {filebrowser_password}")
print("TensorBoard is available inside the WebUI's TensorBoard tab.")

try:
    filebrowser_log_path = repo_dir / "filebrowser.log"
    filebrowser_log = open(filebrowser_log_path, "w", encoding="utf-8")
    filebrowser_process = subprocess.Popen(
        [
            "filebrowser",
            "--database",
            str(filebrowser_database),
            "--root",
            "/kaggle",
            "--address",
            "127.0.0.1",
            "--port",
            "9876",
        ],
        stdout=filebrowser_log,
        stderr=subprocess.STDOUT,
        text=True,
    )
    wait_for_port("127.0.0.1", 9876, filebrowser_process, filebrowser_log_path, timeout=30)

    app_command = [sys.executable, "app.py", "--server-name", "0.0.0.0", "--port", "6969"]
    if Tunnel == "Gradio + LocalTunnel":
        app_command.append("--share")
    app_environment = os.environ.copy()
    app_environment["APPLIO_AUTH_USERNAME"] = app_username
    app_environment["APPLIO_AUTH_PASSWORD"] = app_password
    app_environment["APPLIO_AUTO_PULL_DONE"] = "1"
    app_environment["PYTHONUNBUFFERED"] = "1"
    app_process = subprocess.Popen(
        app_command,
        env=app_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    app_output_thread = threading.Thread(target=stream_process_output, args=(app_process,), daemon=True)
    app_output_thread.start()
    wait_for_port("127.0.0.1", 6969, app_process, timeout=180)

    if Tunnel == "Gradio + LocalTunnel":
        filebrowser_url = start_tunnel(
            ["lt", "--host", "https://loca.lt", "--local-host", "127.0.0.1", "--port", "9876"],
            "filebrowser-localtunnel",
            localtunnel_pattern,
        )
        print(f"LocalTunnel FileBrowser Public URL: {filebrowser_url}")
        print(f"LocalTunnel Password: {localtunnel_password()}")
    elif Tunnel == "LocalTunnel":
        webui_url = start_tunnel(
            ["lt", "--host", "https://loca.lt", "--local-host", "127.0.0.1", "--port", "6969"],
            "webui-localtunnel",
            localtunnel_pattern,
        )
        filebrowser_url = start_tunnel(
            ["lt", "--host", "https://loca.lt", "--local-host", "127.0.0.1", "--port", "9876"],
            "filebrowser-localtunnel",
            localtunnel_pattern,
        )
        print(f"RedPanda WebUI Public URL: {webui_url}")
        print(f"LocalTunnel FileBrowser Public URL: {filebrowser_url}")
        print(f"LocalTunnel Password: {localtunnel_password()}")
    else:
        webui_url = start_tunnel(
            ["hrzn", "tunnel", "http://localhost:6969"],
            "webui-horizon",
            horizon_pattern,
        )
        filebrowser_url = start_tunnel(
            ["hrzn", "tunnel", "http://localhost:9876"],
            "filebrowser-horizon",
            horizon_pattern,
        )
        print(f"RedPanda WebUI Public URL: {webui_url}")
        print(f"Horizon FileBrowser Public URL: {filebrowser_url}")

    if app_process.wait() != 0:
        raise RuntimeError(f"RedPanda WebUI exited with code {app_process.returncode}.")
finally:
    for process in reversed(tunnel_processes):
        stop_process(process)
    stop_process(filebrowser_process)
    stop_process(app_process)
    if app_output_thread is not None:
        app_output_thread.join(timeout=2)
    for log_file in tunnel_logs:
        log_file.close()
    if filebrowser_log is not None:
        filebrowser_log.close()
    for port in ("6969/tcp", "9876/tcp"):
        subprocess.run(["fuser", "-k", port], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)